# 01 – Exploratory Data Analysis

Exploratory data analysis for synthetic grocery receipt line-item data in `data/sample/synthetic_receipts.csv`.

This notebook is structured as follows:

A. Load + schema
B. Data quality checks
C. Core summaries
D. Promo sanity visualization
E. Export artifacts into `reports/`


## A. Load data and inspect schema

In [ ]:
# Standard data and plotting libraries
import os
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

sns.set(style='whitegrid')
pd.set_option('display.max_columns', None)

DATA_PATH = Path('..') / 'data' / 'sample' / 'synthetic_receipts.csv'
DATA_PATH

In [ ]:
# Load CSV
df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
# Parse timestamp to datetime and confirm dtypes
df['timestamp'] = pd.to_datetime(df['timestamp'])

print(df.dtypes)
df.sample(5)

## B. Data quality checks

In [ ]:
# Missing values per column
df.isna().sum().to_frame('missing_count')

In [ ]:
# Optional: duplicate line-items (exact row duplicates)
duplicate_mask = df.duplicated()
print(f'Total duplicate rows: {duplicate_mask.sum()}')

if duplicate_mask.any():
    df[duplicate_mask].head()

In [ ]:
# Negative quantities / refunds (if any)
neg_qty = df[df['quantity'] < 0]
print(f'Rows with negative quantity: {len(neg_qty)}')

if not neg_qty.empty:
    neg_qty.head()

In [ ]:
# Validate line_total == quantity * unit_price
# Spot-check a few rows and compute mismatch count.
calc_line_total = (df['quantity'] * df['unit_price']).round(2)
mismatch_mask = (calc_line_total != df['line_total'].round(2))

print(f'Total mismatched line_total rows: {mismatch_mask.sum()}')

# Show a few mismatches if they exist
if mismatch_mask.any():
    df.loc[mismatch_mask].head()
else:
    df.sample(5)[['transaction_id', 'product_name', 'quantity', 'unit_price', 'line_total']].head()

## C. Core summaries

In [ ]:
# Convenience time-based features
df['date'] = df['timestamp'].dt.date
df['dow'] = df['timestamp'].dt.day_name()
df['hour'] = df['timestamp'].dt.hour
df.head()

In [ ]:
# Top 15 products by revenue
top_products = (
    df.groupby('product_name', as_index=False)
      ['line_total']
      .sum()
)
top_products = top_products.sort_values('line_total', ascending=False).head(15)
top_products

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=top_products, x='line_total', y='product_name', orient='h')
plt.xlabel('Revenue')
plt.ylabel('Product')
plt.title('Top 15 Products by Revenue')
plt.tight_layout()
plt.show()

In [ ]:
# Top categories by revenue
category_revenue = (
    df.groupby('category', as_index=False)['line_total']
      .sum()
      .sort_values('line_total', ascending=False)
)
category_revenue

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=category_revenue, x='category', y='line_total')
plt.xlabel('Category')
plt.ylabel('Revenue')
plt.title('Revenue by Category')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Daily revenue trend (time series)
daily_revenue = (
    df.groupby('date', as_index=False)['line_total']
      .sum()
      .rename(columns={'line_total': 'daily_revenue'})
)
daily_revenue.head()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(daily_revenue['date'], daily_revenue['daily_revenue'], marker='o')
plt.xlabel('Date')
plt.ylabel('Revenue')
plt.title('Daily Revenue Over Time')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## D. Promo sanity visualization

We expect to see temporary price dips for selected promo products and a permanent step change for a price increase.


In [ ]:
# Pick 2 products to track median unit_price over time
promo_products = [
    'Breakfast Cereal 500g',
    'Cola 6-pack',
]

price_median = (
    df[df['product_name'].isin(promo_products)]
      .groupby(['date', 'product_name'], as_index=False)['unit_price']
      .median()
)
price_median.head()

In [ ]:
plt.figure(figsize=(12, 5))
for p in promo_products:
    subset = price_median[price_median['product_name'] == p]
    plt.plot(subset['date'], subset['unit_price'], marker='o', label=p)

plt.xlabel('Date')
plt.ylabel('Median unit price')
plt.title('Daily median unit_price for promo products')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## E. Export artifacts into `reports/`

Save summary tables and a couple of figures to disk so they can be consumed by reports or downstream pipelines.

In [ ]:
tables_dir = Path('..') / 'reports' / 'tables'
figures_dir = Path('..') / 'reports' / 'figures'
tables_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)
tables_dir, figures_dir

In [ ]:
# Export summary tables
top_products.to_csv(tables_dir / 'top_products_by_revenue.csv', index=False)
category_revenue.to_csv(tables_dir / 'category_revenue.csv', index=False)
daily_revenue.to_csv(tables_dir / 'daily_revenue.csv', index=False)

sorted(os.listdir(tables_dir))

In [ ]:
# Save a couple of key plots to figures/
plt.figure(figsize=(10, 5))
sns.barplot(data=top_products, x='line_total', y='product_name', orient='h')
plt.xlabel('Revenue')
plt.ylabel('Product')
plt.title('Top 15 Products by Revenue')
plt.tight_layout()
plt.savefig(figures_dir / 'top_products_by_revenue.png', dpi=150)
plt.close()

plt.figure(figsize=(12, 5))
for p in promo_products:
    subset = price_median[price_median['product_name'] == p]
    plt.plot(subset['date'], subset['unit_price'], marker='o', label=p)
plt.xlabel('Date')
plt.ylabel('Median unit price')
plt.title('Daily median unit_price for promo products')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(figures_dir / 'promo_price_median.png', dpi=150)
plt.close()

sorted(os.listdir(figures_dir))